In [1]:
# Prerequis :
#   uv add kafka-python
#   docker compose -f docker/docker-compose.yml up -d
# Necessite un VRAI broker Kafka en cours d'execution (pas juste un
# acces reseau) -- bloque dans le sandbox utilise pour ecrire ce code
# (aucun broker disponible), a lancer chez toi une fois Docker demarre.
# Utilise les fonctions reelles de src/realtime/streaming.py

# --- BLOC 1 : verifier que le broker repond ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from realtime.streaming import (
    consume_and_analyze,
    create_consumer,
    create_producer,
    publish_review,
)

TOPIC = "customer-reviews-test"

producteur = create_producer()
print("Producteur cree, connecte a localhost:9092")

Producteur cree, connecte a localhost:9092


/tmp/ipykernel_118925/465759162.py:18: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producteur = create_producer()


In [2]:
# --- BLOC 2 : publier quelques avis, comme s'ils arrivaient en direct ---
avis_a_publier = [
    "The delivery was super fast, arrived the next day",
    "Customer service was rude and unhelpful this time",
    "Great product quality but the price is a bit high",
]

for i, avis in enumerate(avis_a_publier, start=1):
    publish_review(producteur, TOPIC, avis, review_id=i)
    print(f"Avis {i} publie : {avis}")

producteur.close()

Avis 1 publie : The delivery was super fast, arrived the next day
Avis 2 publie : Customer service was rude and unhelpful this time
Avis 3 publie : Great product quality but the price is a bit high


In [3]:
# --- BLOC 3 : consommer et analyser (reutilise le modele ABSA, Phase 9) ---
from aspect_sentiment.absa import load_absa_classifier

model, tokenizer = load_absa_classifier()

consommateur = create_consumer(TOPIC)
resultats = consume_and_analyze(
    consommateur, model, tokenizer, max_messages=len(avis_a_publier)
)
consommateur.close()

print(f"\n{len(resultats)} avis consommes et analyses :\n")
for r in resultats:
    print(f"Avis #{r['review_id']} : {r['text']}")
    for aspect, sentiment in r["aspect_sentiments"].items():
        print(f"    {aspect:20} -> {sentiment}")
    print()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



3 avis consommes et analyses :

Avis #1 : The delivery was super fast, arrived the next day
    delivery             -> neutral
    day                  -> neutral

Avis #2 : Customer service was rude and unhelpful this time
    Customer service     -> neutral

Avis #3 : Great product quality but the price is a bit high
    Great product quality -> neutral
    price                -> neutral



In [4]:
# --- BLOC 4 : verifier l'ordre de traitement (FIFO garanti par Kafka) ---
ids_recus = [r["review_id"] for r in resultats]
print("Ordre de reception :", ids_recus)
assert ids_recus == list(range(1, len(avis_a_publier) + 1)), (
    "Kafka garantit l'ordre FIFO au sein d'une meme partition -- "
    "les avis doivent etre recus dans l'ordre exact de publication"
)
print("OK : les avis sont recus dans l'ordre exact de publication")

Ordre de reception : [1, 2, 3]
OK : les avis sont recus dans l'ordre exact de publication


In [ ]:
# --- BLOC 5 : simuler un consommateur qui demarre APRES coup ---
# verifie auto_offset_reset="earliest" (voir streaming.py) : meme si
# le consommateur se connecte APRES que les avis aient ete publies, il
# doit quand meme tous les recevoir depuis le debut, pas les rater
import time

producteur2 = create_producer()
publish_review(
    producteur2,
    "customer-reviews-test-2",
    "Avis publie AVANT la connexion du consommateur",
    review_id=99,
)
producteur2.close()

time.sleep(1)  # laisse le temps au message d'etre bien enregistre par Kafka

consommateur2 = create_consumer("customer-reviews-test-2")
resultats2 = consume_and_analyze(consommateur2, model, tokenizer, max_messages=1)
consommateur2.close()

print("\nMessage recu malgre une connexion tardive :", resultats2[0]["text"])
assert resultats2[0]["review_id"] == 99
print("OK : auto_offset_reset='earliest' fonctionne comme prevu")

/tmp/ipykernel_118925/534949174.py:5: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producteur2 = create_producer()



Message recu malgre une connexion tardive : Avis publie AVANT la connexion du consommateur
OK : auto_offset_reset='earliest' fonctionne comme prevu


In [6]:
# --- BLOC 6 : debit -- combien d'avis par seconde peut-on traiter ? ---
import time

producteur3 = create_producer()
n_avis = 20
debut_publication = time.time()
for i in range(n_avis):
    publish_review(
        producteur3,
        "customer-reviews-perf",
        avis_a_publier[i % len(avis_a_publier)],
        review_id=i,
    )
producteur3.close()
duree_publication = time.time() - debut_publication
print(f"\n{n_avis} avis publies en {duree_publication:.2f}s")

consommateur3 = create_consumer("customer-reviews-perf")
debut_consommation = time.time()
resultats3 = consume_and_analyze(consommateur3, model, tokenizer, max_messages=n_avis)
consommateur3.close()
duree_consommation = time.time() - debut_consommation

print(f"{n_avis} avis consommes ET analyses en {duree_consommation:.2f}s")
debit = n_avis / duree_consommation
print(f"Debit : {debit:.2f} avis/seconde " f"(goulot = l'analyse ABSA, pas Kafka)")

/tmp/ipykernel_118925/3615886525.py:4: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producteur3 = create_producer()



20 avis publies en 0.27s
20 avis consommes ET analyses en 0.89s
Debit : 22.56 avis/seconde (goulot = l'analyse ABSA, pas Kafka)


In [8]:
# --- BLOC 7 : voir REELLEMENT l'arrivee etalee dans le temps ---
# Contrairement aux blocs precedents (publication en rafale, pour
# tester la performance), ce bloc simule des avis qui arrivent
# REELLEMENT les uns apres les autres, avec un delai, et affiche
# l'heure exacte de publication ET de reception -- pour VOIR
# concretement ce que "flux en temps reel" veut dire.
import datetime
import time

TOPIC_TEMPOREL = "customer-reviews-temporel"
avis_realistes = [
    "The delivery was incredibly fast this time",
    "Customer service resolved my issue immediately",
    "Product arrived damaged, very disappointed",
    "Great value for the price, will buy again",
]

producteur4 = create_producer()

print("=== Publication avec delai realiste (3 secondes entre chaque) ===\n")
for i, avis in enumerate(avis_realistes, start=1):
    heure_publication = datetime.datetime.now().strftime("%H:%M:%S")
    publish_review(producteur4, TOPIC_TEMPOREL, avis, review_id=i)
    print(f"[{heure_publication}] PUBLIE  : avis #{i} -- {avis}")
    time.sleep(10)  # attend 3 secondes avant le prochain avis

producteur4.close()

/tmp/ipykernel_118925/2592123941.py:18: DeprecationWarning: value_serializer does not implement kafka.serializer.Serializer
  producteur4 = create_producer()


=== Publication avec delai realiste (3 secondes entre chaque) ===

[12:23:55] PUBLIE  : avis #1 -- The delivery was incredibly fast this time
[12:24:05] PUBLIE  : avis #2 -- Customer service resolved my issue immediately
[12:24:15] PUBLIE  : avis #3 -- Product arrived damaged, very disappointed
[12:24:25] PUBLIE  : avis #4 -- Great value for the price, will buy again


In [13]:
# --- BLOC 8 : consommer ce meme flux, en notant l'heure de RECEPTION ---
# Note : ce bloc tourne APRES le Bloc 7 ci-dessus, donc tous les avis
# sont deja dans Kafka au moment ou on commence a consommer -- pour
# voir un vrai DECALAGE entre publication et reception (comme le
# ferait un consommateur qui tourne en permanence, Phase 14), il
# faudrait lancer ce Bloc 8 dans un terminal SEPARE, EN PARALLELE du
# Bloc 7 -- essaie-le si tu veux voir l'effet complet.
consommateur4 = create_consumer(TOPIC_TEMPOREL)

print("\n=== Consommation, avec heure de reception ===\n")
for count, message in enumerate(consommateur4):
    heure_reception = datetime.datetime.now().strftime("%H:%M:%S")
    review = message.value
    print(
        f"[{heure_reception}] RECU    : avis #{review['review_id']} "
        f"-- {review['text']}"
    )
    if count + 1 >= len(avis_realistes):
        break
consommateur4.close()


=== Consommation, avec heure de reception ===

[12:25:12] RECU    : avis #1 -- The delivery was incredibly fast this time
[12:25:12] RECU    : avis #2 -- Customer service resolved my issue immediately
[12:25:12] RECU    : avis #3 -- Product arrived damaged, very disappointed
[12:25:12] RECU    : avis #4 -- Great value for the price, will buy again
